In [2]:
"""
Combined MF-DFA Analysis
=========================
1. Pearson correlation between F_{+q}(s) and F_{-q}(s)  [q = 1..30]
2. 6-panel multifractal analysis (H(q), tau(q), alpha(q), f(alpha))

Both run on:
  (A) Real log-returns
  (B) Shuffled surrogate log-returns  (phase-randomised null hypothesis)

Output directories:
  mf_corr_plots/                          — Pearson correlation plots   (real data)
  mf_plots/                               — 6-panel multifractal plots  (real data)
  shuffled_trials/mf_corr_plots/          — Pearson correlation plots   (shuffled)
  shuffled_trials/mf_plots/               — 6-panel multifractal plots  (shuffled)
  shuffled_trials/mf_corr_plots_comparison/ — Overlay: real vs surrogate (Pearson)
"""

import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.signal import savgol_filter
import os, warnings

warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)   # reproducible shuffles

# ─────────────────────────────────────────────────────────────────────────────
# PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
SECTORS = {
    "Tech": {
        "tickers": ["AAPL", "MSFT", "GOOG", "AMZN", "NVDA",
                    "META", "TSLA", "NFLX", "AMD", "INTC"],
        "colors":  ["#1f77b4", "#4e9fd4", "#2196f3", "#0d47a1", "#17becf",
                    "#00bcd4", "#006994", "#4fc3f7", "#0288d1", "#80d8ff"],
    },
    "Financials": {
        "tickers": ["JPM", "BAC", "GS", "MS", "WFC",
                    "BLK", "C", "AXP", "V", "MA"],
        "colors":  ["#2ca02c", "#27ae60", "#1abc9c", "#16a085", "#8db600",
                    "#4caf50", "#388e3c", "#00897b", "#6b8e23", "#a5d6a7"],
    },
    "Healthcare & Consumer": {
        "tickers": ["JNJ", "PFE", "UNH", "MRK", "ABBV",
                    "PG", "KO", "PEP", "WMT", "COST"],
        "colors":  ["#9467bd", "#8e44ad", "#6c3483", "#7c3aed", "#a855f7",
                    "#ba68c8", "#7b1fa2", "#ce93d8", "#76448a", "#e1bee7"],
    },
    "Energy & Industrials": {
        "tickers": ["XOM", "CVX", "COP", "BA", "CAT",
                    "GE", "HON", "LMT", "RTX", "UPS"],
        "colors":  ["#ff7f0e", "#e8631a", "#f39c12", "#d35400", "#ff6b6b",
                    "#e64a19", "#bf360c", "#ff8f00", "#f57c00", "#ffcc02"],
    },
    "Commodities & Crypto": {
        "tickers": ["GC=F", "SI=F", "CL=F", "BTC-USD", "ETH-USD"],
        "colors":  ["#bcbd22", "#f1c40f", "#ffd600", "#ffab00", "#ffe57f"],
    },
    "International ETFs": {
        "tickers": ["EWJ", "EWZ", "FXI", "EWG", "EWU"],
        "colors":  ["#d62728", "#c0392b", "#e74c3c", "#b71c1c", "#ef9a9a"],
    },
}

# Flat ticker list for 6-panel plots
ALL_TICKERS = [t for s in SECTORS.values() for t in s["tickers"]]

START      = "2010-01-01"
END        = "2026-04-11"
Q_CORR     = np.arange(1, 31, dtype=float)        # q for Pearson plots
Q_MFSPEC   = np.arange(-30, 31, 5, dtype=float)   # q for 6-panel spectrum
SCALE_MIN  = 20
SCALE_MAX  = 800
N_SCALES   = 30
POLY_ORD   = 1
OVERLAP    = 0.75

PAPER_COLORS = [
    "#d62728", "#2ca02c", "#1f77b4", "#000000", "#9467bd",
    "#ff7f0e", "#8c564b", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f",
]

# Output directories
for d in ["mf_corr_plots", "mf_plots",
          "shuffled_trials/mf_corr_plots",
          "shuffled_trials/mf_plots",
          "shuffled_trials/mf_corr_plots_comparison"]:
    os.makedirs(d, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# DATA PREPARATION
# ─────────────────────────────────────────────────────────────────────────────
def prepare_returns(price):
    price = price[price > 0]
    return np.diff(np.log(price))


def shuffle_surrogate(returns):
    """Randomly shuffle returns — destroys temporal correlations / multifractality."""
    s = returns.copy()
    rng.shuffle(s)
    return s


def integrate(x):
    """DFA profile: cumulative sum of mean-centred series."""
    return np.cumsum(x - x.mean())


def _overlap_starts(N, s, overlap):
    step = max(1, int(s * (1.0 - overlap)))
    return np.arange(0, N - s + 1, step)


# ─────────────────────────────────────────────────────────────────────────────
# MF-DFA CORE
# ─────────────────────────────────────────────────────────────────────────────
def _compute_rms_per_scale(x, scale_min, scale_max, n_scales, poly_ord, overlap):
    """
    Returns (scales, rms_dict) where rms_dict[i] = array of segment RMS at scales[i].
    Works on the integrated profile of x.
    """
    y  = integrate(x)
    N  = len(y)
    scale_max = min(scale_max, N // 4)
    scale_min = max(scale_min, poly_ord + 2)

    scales = np.unique(
        np.logspace(np.log10(scale_min), np.log10(scale_max), n_scales).astype(int)
    )
    scales = scales[scales >= scale_min]

    rms_per_scale = {}
    for i, s in enumerate(scales):
        starts = _overlap_starts(N, s, overlap)
        if len(starts) < 4:
            rms_per_scale[i] = np.array([])
            continue
        t   = np.arange(s)
        rms = []
        for st in starts:
            seg  = y[st : st + s]
            coef = np.polyfit(t, seg, poly_ord)
            r    = np.sqrt(np.mean((seg - np.polyval(coef, t)) ** 2))
            if r > 0:
                rms.append(r)
        rms_per_scale[i] = np.asarray(rms)

    return scales, rms_per_scale


def mfdfa_symmetric(x, q_vals, scale_min, scale_max, n_scales,
                    poly_ord=1, overlap=0.75):
    """
    Returns (scales, Fqs_pos, Fqs_neg) for Pearson correlation plots.
    q_vals should be positive (1..30); both +q and -q fluctuations are computed.
    """
    scales, rms_dict = _compute_rms_per_scale(
        x, scale_min, scale_max, n_scales, poly_ord, overlap)

    Fqs_pos = np.full((len(q_vals), len(scales)), np.nan)
    Fqs_neg = np.full((len(q_vals), len(scales)), np.nan)

    for i, s_idx in enumerate(range(len(scales))):
        rms = rms_dict[s_idx]
        if len(rms) == 0:
            continue
        for j, q in enumerate(q_vals):
            Fqs_pos[j, i] = (np.mean(rms **  q)) ** (1.0 /  q)
            Fqs_neg[j, i] = (np.mean(rms ** -q)) ** (1.0 / -q)

    return scales, Fqs_pos, Fqs_neg


def mfdfa_spectrum(x, q_vals, scale_min, scale_max, n_scales,
                   poly_ord=1, overlap=0.75):
    """
    Returns (scales, Fqs, hq) for 6-panel multifractal spectrum.
    q_vals spans negative to positive.
    """
    scales, rms_dict = _compute_rms_per_scale(
        x, scale_min, scale_max, n_scales, poly_ord, overlap)

    Fqs = np.full((len(q_vals), len(scales)), np.nan)

    for i in range(len(scales)):
        rms = rms_dict[i]
        if len(rms) == 0:
            continue
        for j, q in enumerate(q_vals):
            if q == 0:
                Fqs[j, i] = np.exp(0.5 * np.mean(np.log(rms ** 2)))
            else:
                Fqs[j, i] = (np.mean(rms ** q)) ** (1.0 / q)

    hq = np.full(len(q_vals), np.nan)
    for j in range(len(q_vals)):
        row  = Fqs[j]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() >= 4:
            hq[j] = np.polyfit(np.log(scales[mask]), np.log(row[mask]), 1)[0]

    return scales, Fqs, hq


# ─────────────────────────────────────────────────────────────────────────────
# MULTIFRACTAL SPECTRUM
# ─────────────────────────────────────────────────────────────────────────────
def multifractal_spectrum(q_vals, hq):
    tau_q = q_vals * hq - 1.0
    wl = min(5, len(tau_q) if len(tau_q) % 2 == 1 else len(tau_q) - 1)
    tau_smooth = (savgol_filter(tau_q, window_length=wl, polyorder=2)
                  if len(tau_q) >= 5 else tau_q)
    alpha   = np.gradient(tau_smooth, q_vals)
    D_alpha = q_vals * alpha - tau_smooth
    return tau_q, alpha, D_alpha


# ─────────────────────────────────────────────────────────────────────────────
# PEARSON HELPER
# ─────────────────────────────────────────────────────────────────────────────
def pearson_corr(a, b):
    mask = np.isfinite(a) & np.isfinite(b) & (a > 0) & (b > 0)
    if mask.sum() < 3:
        return np.nan
    a_, b_ = np.log(a[mask]), np.log(b[mask])
    cov   = np.mean((a_ - a_.mean()) * (b_ - b_.mean()))
    denom = a_.std() * b_.std()
    if denom == 0:
        return np.nan
    return cov / denom


# ─────────────────────────────────────────────────────────────────────────────
# PLOT: PEARSON CORRELATION (single dataset — real or surrogate)
# ─────────────────────────────────────────────────────────────────────────────
def plot_pearson_by_sector(corr_data, out_dir, title_suffix="", ylim_tight=True):
    """
    corr_data: dict  sector_name -> list of (ticker, color, corr_vals array)
    Produces both tight (0.90–1.005) and full-range (-0.1–1.05) versions.
    """
    n_sectors = len(corr_data)
    ncols = 2
    nrows = int(np.ceil(n_sectors / ncols))

    for full_range in [False, True]:
        fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows),
                                 sharex=True, sharey=True)
        axes = axes.flatten()

        for ax, (sector_name, entries) in zip(axes, corr_data.items()):
            for ticker, color, corr_vals in entries:
                valid = np.isfinite(corr_vals)
                ax.plot(Q_CORR[valid], corr_vals[valid],
                        "o-", color=color, linewidth=1.8,
                        markersize=4.5, label=ticker)

            ax.set_title(sector_name, fontsize=13, fontweight="bold")
            ax.set_xlabel("|q|", fontsize=11)
            ax.set_ylabel("Pearson $r$", fontsize=11)
            ax.set_xticks(Q_CORR[::2])
            ax.set_xlim(1, 30)

            if full_range:
                ax.set_ylim(-0.1, 1.05)
                ax.yaxis.set_major_locator(plt.MultipleLocator(0.10))
            else:
                ax.set_ylim(0.90, 1.005)
                ax.yaxis.set_major_locator(plt.MultipleLocator(0.01))

            ax.axhline(1.0, color="grey", lw=0.8, ls="--", alpha=0.4)
            ax.axhline(0.0, color="grey", lw=0.8, ls=":",  alpha=0.4)
            ax.tick_params(direction="in", which="both", top=True, right=True)
            ax.legend(fontsize=8.5, ncol=2, loc="lower left", framealpha=0.9)

        for ax in axes[n_sectors:]:
            ax.set_visible(False)

        range_tag  = "fullrange" if full_range else "tight"
        range_title = "[Full Range]" if full_range else ""
        fig.suptitle(
            f"Pearson correlation between $F_{{+q}}(s)$ and $F_{{-q}}(s)$\n"
            f"MF-DFA · Log-Returns · q = 1 to 30 · By Sector"
            f"{title_suffix} {range_title}",
            fontsize=14, fontweight="bold", y=1.01
        )
        plt.tight_layout()
        fname = f"{out_dir}/mfdfa_pearson_by_sector_{range_tag}.png"
        plt.savefig(fname, dpi=130, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT: COMPARISON OVERLAY (real vs surrogate)
# ─────────────────────────────────────────────────────────────────────────────
def plot_pearson_comparison(real_data, surr_data, out_dir):
    """
    Overlay real (solid) vs surrogate (dashed) Pearson curves per sector.
    """
    n_sectors = len(real_data)
    ncols = 2
    nrows = int(np.ceil(n_sectors / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows),
                             sharex=True, sharey=True)
    axes = axes.flatten()

    for ax, sector_name in zip(axes, real_data.keys()):
        real_entries = real_data[sector_name]
        surr_entries = {t: cv for t, _, cv in surr_data[sector_name]}

        for ticker, color, real_cv in real_entries:
            surr_cv = surr_entries.get(ticker)
            valid_r = np.isfinite(real_cv)
            ax.plot(Q_CORR[valid_r], real_cv[valid_r],
                    "o-", color=color, linewidth=1.8,
                    markersize=4.0, label=ticker)
            if surr_cv is not None:
                valid_s = np.isfinite(surr_cv)
                ax.plot(Q_CORR[valid_s], surr_cv[valid_s],
                        "--", color=color, linewidth=1.2,
                        markersize=0, alpha=0.55)

        # Dummy legend entries for line styles
        ax.plot([], [], "k-",  linewidth=1.8, label="Real")
        ax.plot([], [], "k--", linewidth=1.2, alpha=0.7, label="Surrogate")

        ax.set_title(sector_name, fontsize=13, fontweight="bold")
        ax.set_xlabel("|q|", fontsize=11)
        ax.set_ylabel("Pearson $r$", fontsize=11)
        ax.set_xticks(Q_CORR[::2])
        ax.set_xlim(1, 30)
        ax.set_ylim(0.85, 1.005)
        ax.yaxis.set_major_locator(plt.MultipleLocator(0.02))
        ax.axhline(1.0, color="grey", lw=0.8, ls="--", alpha=0.4)
        ax.tick_params(direction="in", which="both", top=True, right=True)
        ax.legend(fontsize=7.5, ncol=2, loc="lower left", framealpha=0.9)

    for ax in axes[n_sectors:]:
        ax.set_visible(False)

    fig.suptitle(
        "Pearson correlation: Real (solid) vs Shuffled Surrogate (dashed)\n"
        "MF-DFA · Log-Returns · q = 1 to 30 · By Sector",
        fontsize=14, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    fname = f"{out_dir}/mfdfa_pearson_real_vs_surrogate.png"
    plt.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# PLOT: 6-PANEL MULTIFRACTAL
# ─────────────────────────────────────────────────────────────────────────────
def plot_6panel(ticker, out_dir, raw_series,
                scales, Fqs, hq, q_vals,
                tau_q, alpha, D_alpha,
                label_suffix=""):
    valid   = np.isfinite(hq)
    colors  = (PAPER_COLORS * 4)[: len(q_vals)]

    fig, axs = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(
        f"Multifractal analysis · {ticker} · MF-DFA · Log-Returns{label_suffix}",
        fontsize=13, fontweight="bold", y=1.01,
    )

    # (a) raw series
    ax = axs[0, 0]
    ax.plot(raw_series, color="#c0392b", linewidth=0.65)
    ax.set_xlabel("$t$", fontsize=11)
    ax.set_ylabel("$X(t)$", fontsize=11)
    ax.set_title("(a) Time series", fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.4g"))

    # (b) F_q(s) log-log
    ax = axs[0, 1]
    legend_handles = []
    for idx_q, (q, c) in enumerate(zip(q_vals, colors)):
        row  = Fqs[idx_q]
        mask = np.isfinite(row) & (row > 0)
        if mask.sum() < 2:
            continue
        h, = ax.loglog(scales[mask], row[mask], "o-",
                       color=c, markersize=3.5, linewidth=1.0)
        legend_handles.append((h, f"$q={int(q)}$"))
    ax.set_xlabel("$s$", fontsize=11)
    ax.set_ylabel("$F_q(s)$", fontsize=11)
    ax.set_title("(b) Fluctuation functions $F_q(s)$", fontsize=11)
    step  = max(1, len(legend_handles) // 8)
    shown = legend_handles[::step]
    ax.legend([h for h, _ in shown], [l for _, l in shown],
              fontsize=7, ncol=1, loc="best",
              handlelength=1.2, handletextpad=0.4)

    # (c) H(q)
    ax = axs[0, 2]
    ax.plot(q_vals[valid], hq[valid], "-", color="#1f4e8c", linewidth=2)
    ax.set_xlabel("$q$", fontsize=11)
    ax.set_ylabel("$H(q)$", fontsize=11)
    ax.set_title("(c) Generalised Hurst exponent $H(q)$", fontsize=11)
    ax.axhline(0.5, color="grey", lw=0.8, ls="--", alpha=0.6, label="H=0.5")
    ax.axhline(1.0, color="blue", lw=0.8, ls=":",  alpha=0.6, label="H=1.0")
    ax.legend(fontsize=7)

    # (d) tau(q)
    ax = axs[1, 0]
    ax.plot(q_vals[valid], tau_q[valid], "-", color="#1f4e8c", linewidth=2)
    ax.set_xlabel("$q$", fontsize=11)
    ax.set_ylabel(r"$\tau(q)$", fontsize=11)
    ax.set_title(r"(d) Scaling exponent $\tau(q)$", fontsize=11)
    ax.axhline(0, color="grey", lw=0.8, ls="--", alpha=0.6)

    # (e) alpha(q)
    ax = axs[1, 1]
    valid_s = np.isfinite(alpha) & np.isfinite(D_alpha)
    ax.plot(q_vals[valid_s], alpha[valid_s], "-", color="#1f4e8c", linewidth=2)
    ax.set_xlabel("$q$", fontsize=11)
    ax.set_ylabel(r"$\alpha(q)$", fontsize=11)
    ax.set_title(r"(e) Singularity strength $\alpha(q)$", fontsize=11)

    # (f) f(alpha)
    ax = axs[1, 2]
    ax.plot(alpha[valid_s], D_alpha[valid_s], "-", color="#1f4e8c", linewidth=2)
    q0_idx = np.argmin(np.abs(q_vals))
    if np.isfinite(alpha[q0_idx]) and np.isfinite(D_alpha[q0_idx]):
        ax.plot(alpha[q0_idx], D_alpha[q0_idx], "o",
                color="#c0392b", markersize=6, zorder=5)
    ax.set_xlabel(r"$\alpha$", fontsize=11)
    ax.set_ylabel(r"$f(\alpha)$", fontsize=11)
    ax.set_title(r"(f) Multifractal spectrum $f(\alpha)$", fontsize=11)

    for ax in axs.ravel():
        ax.tick_params(direction="in", which="both", top=True, right=True)
        ax.spines[["top", "right"]].set_visible(True)

    delta_alpha = np.nan
    if valid_s.sum() >= 2:
        delta_alpha = alpha[valid_s].max() - alpha[valid_s].min()
        fig.text(0.5, -0.01,
                 fr"Multifractal width  $\Delta\alpha = {delta_alpha:.4f}$",
                 ha="center", fontsize=10, color="grey")

    plt.tight_layout()
    safe = ticker.replace("=", "_").replace("-", "_")
    fname = f"{out_dir}/{safe}_MF_DFA_returns{label_suffix.replace(' ', '_')}.png"
    plt.savefig(fname, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"    Saved → {fname}   Δα={delta_alpha:.4f}" if np.isfinite(delta_alpha)
          else f"    Saved → {fname}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("  PHASE 1 — Download data & compute both real and surrogate results")
print("=" * 70)

# Storage for Pearson plots  {sector_name: [(ticker, color, corr_vals), ...]}
real_corr = {s: [] for s in SECTORS}
surr_corr = {s: [] for s in SECTORS}

for sector_name, sector in SECTORS.items():
    print(f"\n── Sector: {sector_name} ──")

    for ticker, color in zip(sector["tickers"], sector["colors"]):
        print(f"  {ticker}...", end=" ", flush=True)

        data  = yf.download(ticker, start=START, end=END, progress=False)
        close = data["Close"].dropna().values.ravel().astype(float)

        if len(close) < 200:
            print("skipped (insufficient data).")
            continue

        returns  = prepare_returns(close)
        shuffled = shuffle_surrogate(returns)

        s_min = max(SCALE_MIN, POLY_ORD + 2)
        s_max = min(SCALE_MAX, len(returns) // 4)

        # ── Pearson correlation (q=1..30) ──────────────────────────────────
        scales_r, Fp_r, Fn_r = mfdfa_symmetric(
            returns,  Q_CORR, s_min, s_max, N_SCALES, POLY_ORD, OVERLAP)
        scales_s, Fp_s, Fn_s = mfdfa_symmetric(
            shuffled, Q_CORR, s_min, s_max, N_SCALES, POLY_ORD, OVERLAP)

        cv_real = np.array([pearson_corr(Fp_r[j], Fn_r[j])
                            for j in range(len(Q_CORR))])
        cv_surr = np.array([pearson_corr(Fp_s[j], Fn_s[j])
                            for j in range(len(Q_CORR))])

        real_corr[sector_name].append((ticker, color, cv_real))
        surr_corr[sector_name].append((ticker, color, cv_surr))

        # ── 6-panel: real ──────────────────────────────────────────────────
        sc_r, Fqs_r, hq_r = mfdfa_spectrum(
            returns, Q_MFSPEC, s_min, s_max, N_SCALES, POLY_ORD, OVERLAP)
        tau_r, al_r, Da_r = multifractal_spectrum(Q_MFSPEC, hq_r)
        plot_6panel(ticker, "mf_plots", returns,
                    sc_r, Fqs_r, hq_r, Q_MFSPEC,
                    tau_r, al_r, Da_r)

        # ── 6-panel: surrogate ─────────────────────────────────────────────
        sc_s, Fqs_s, hq_s = mfdfa_spectrum(
            shuffled, Q_MFSPEC, s_min, s_max, N_SCALES, POLY_ORD, OVERLAP)
        tau_s, al_s, Da_s = multifractal_spectrum(Q_MFSPEC, hq_s)
        plot_6panel(ticker, "shuffled_trials/mf_plots", shuffled,
                    sc_s, Fqs_s, hq_s, Q_MFSPEC,
                    tau_s, al_s, Da_s,
                    label_suffix=" [Shuffled Surrogate]")

        print(f"real Δr=[{np.nanmin(cv_real):.3f},{np.nanmax(cv_real):.3f}]  "
              f"surr Δr=[{np.nanmin(cv_surr):.3f},{np.nanmax(cv_surr):.3f}]")

# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  PHASE 2 — Saving Pearson correlation sector plots")
print("=" * 70)

print("\n[Real data]")
plot_pearson_by_sector(real_corr, "mf_corr_plots",     title_suffix="")

print("\n[Shuffled surrogate]")
plot_pearson_by_sector(surr_corr, "shuffled_trials/mf_corr_plots",
                       title_suffix=" [Shuffled Surrogate]")

print("\n[Comparison overlay]")
plot_pearson_comparison(real_corr, surr_corr, "shuffled_trials/mf_corr_plots_comparison")

print("\n" + "=" * 70)
print("  ALL DONE")
print("  mf_corr_plots/                            — Pearson correlation (real data)")
print("  mf_plots/                                 — 6-panel multifractal (real data)")
print("  shuffled_trials/mf_corr_plots/            — Pearson correlation (shuffled)")
print("  shuffled_trials/mf_plots/                 — 6-panel multifractal (shuffled)")
print("  shuffled_trials/mf_corr_plots_comparison/ — Overlay real vs surrogate")
print("=" * 70)

  PHASE 1 — Download data & compute both real and surrogate results

── Sector: Tech ──
  AAPL...     Saved → mf_plots/AAPL_MF_DFA_returns.png   Δα=0.4913
    Saved → shuffled_trials/mf_plots/AAPL_MF_DFA_returns_[Shuffled_Surrogate].png   Δα=0.2904
real Δr=[0.970,1.000]  surr Δr=[0.985,0.999]
  MSFT...     Saved → mf_plots/MSFT_MF_DFA_returns.png   Δα=0.4882
    Saved → shuffled_trials/mf_plots/MSFT_MF_DFA_returns_[Shuffled_Surrogate].png   Δα=0.3724
real Δr=[0.923,1.000]  surr Δr=[0.971,1.000]
  GOOG...     Saved → mf_plots/GOOG_MF_DFA_returns.png   Δα=0.3538
    Saved → shuffled_trials/mf_plots/GOOG_MF_DFA_returns_[Shuffled_Surrogate].png   Δα=0.3392
real Δr=[0.969,0.999]  surr Δr=[0.974,1.000]
  AMZN...     Saved → mf_plots/AMZN_MF_DFA_returns.png   Δα=0.4870
    Saved → shuffled_trials/mf_plots/AMZN_MF_DFA_returns_[Shuffled_Surrogate].png   Δα=0.3340
real Δr=[0.974,1.000]  surr Δr=[0.978,1.000]
  NVDA...     Saved → mf_plots/NVDA_MF_DFA_returns.png   Δα=0.1812
    Saved → shuffled_


1 Failed download:
['C']: Timeout('Failed to perform, curl: (28) Operation timed out after 10008 milliseconds with 48355 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')


skipped (insufficient data).
  AXP...     Saved → mf_plots/AXP_MF_DFA_returns.png   Δα=0.6109
    Saved → shuffled_trials/mf_plots/AXP_MF_DFA_returns_[Shuffled_Surrogate].png   Δα=0.4728
real Δr=[0.975,1.000]  surr Δr=[0.976,1.000]
  V...     Saved → mf_plots/V_MF_DFA_returns.png   Δα=0.5382
    Saved → shuffled_trials/mf_plots/V_MF_DFA_returns_[Shuffled_Surrogate].png   Δα=0.4711
real Δr=[0.837,1.000]  surr Δr=[0.986,1.000]
  MA...     Saved → mf_plots/MA_MF_DFA_returns.png   Δα=0.6367
    Saved → shuffled_trials/mf_plots/MA_MF_DFA_returns_[Shuffled_Surrogate].png   Δα=0.2955
real Δr=[0.752,0.999]  surr Δr=[0.979,0.999]

── Sector: Healthcare & Consumer ──
  JNJ...     Saved → mf_plots/JNJ_MF_DFA_returns.png   Δα=0.7528
    Saved → shuffled_trials/mf_plots/JNJ_MF_DFA_returns_[Shuffled_Surrogate].png   Δα=0.3317
real Δr=[0.398,1.000]  surr Δr=[0.975,1.000]
  PFE...     Saved → mf_plots/PFE_MF_DFA_returns.png   Δα=0.5810
    Saved → shuffled_trials/mf_plots/PFE_MF_DFA_returns_[Shuffled_